# Ch 4. 오픈 웨이트 SLM 풍경 — 크기 · dense · MoE

**오픈 웨이트 SLM이 왜 135M / 360M / 1.7B / 3B / 7B 같은 크기로 나오는지, dense vs MoE 구분을 손으로 확인한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part1/ch04_open_weight_landscape.ipynb)

In [ ]:
# !pip install -q transformers torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device 확인
if torch.cuda.is_available():
    device = "cuda"
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {vram:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Apple Silicon MPS")
else:
    device = "cpu"
    print("CPU only")

print(f"Device: {device}")

## dense vs MoE — 파라미터 수 산수 먼저

MoE(Mixture of Experts)의 핵심:
- FFN 자리에 **N개의 expert**와 **router**를 둠
- 각 토큰마다 router가 N개 중 **k개만** 골라 통과 (보통 k=2)
- "활성 파라미터"가 전체 파라미터의 일부

**핵심 함의**:
- **메모리는 전체** 파라미터 기준
- **연산·속도는 활성** 파라미터 기준
- MoE는 "메모리는 비싸고 속도는 싼" 트레이드오프

In [ ]:
# MoE 파라미터 산수
def moe_params(n_experts, expert_size_b, shared_params_b=0):
    """전체 파라미터 = n_experts * expert_size + shared_params"""
    total = n_experts * expert_size_b + shared_params_b
    return total

# Mixtral 8×7B 예시 (실제로는 non-expert 공유 파라미터가 있음)
# 8 expert * 7B = 56B 이지만 실제는 47B — attention 같은 shared params 때문
models = [
    ("Mixtral 8x7B (MoE)", 47, 13, "47B 수준", "13B 수준"),
    ("Phi-3.5-MoE",         42,  6.6, "42B 수준", "6.6B 수준"),
    ("DeepSeek-V3",        671, 37, "671B 수준", "37B 수준"),
    ("Llama 3 70B (dense)", 70, 70, "140GB",     "70B 수준"),
]

print(f"{'모델':20s}  {'전체':>8s}  {'활성':>8s}  {'메모리':>12s}  {'속도':>10s}")
print("-" * 68)
for name, total, active, mem, speed in models:
    print(f"  {name:20s}  {total:6.0f}B  {active:6.1f}B  {mem:>12s}  {speed:>10s}")

## 최소 예제 — 같은 작업을 5 모델에 던지기

같은 한국어 요약 프롬프트를 dense 5개에 던져 능력 곡선을 본다.

**관찰 가이드**:
- 135M~360M: 한국어 자체가 깨짐 (학습 데이터 영어 위주)
- SmolLM2-1.7B: 한국어 가능, 요약 품질은 들쑥날쑥
- Qwen 2.5-0.5B / 1.5B: 한국어 능력 큰 차이 — Qwen 시리즈가 다국어 학습 비중이 다름

**결론적 직관**: "한국어 + 작은 모델"이면 Qwen 2.5 가족이 SmolLM2보다 보통 낫다.

In [ ]:
# size_sweep.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

prompt = """다음 글을 두 문장으로 요약해줘:
"오픈 웨이트 SLM들이 135M, 360M, 1.7B 같은 크기로 나오는 이유는 디바이스 사다리에 맞췄기 때문이다.
모바일(4GB RAM)에서는 2B 이하, 노트북(16GB)에서는 7B 이하, 단일 A100(80GB)에서는 30B 이하가 적합하다."
"""

models_to_test = [
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-360M",
    "HuggingFaceTB/SmolLM2-1.7B",
    "Qwen/Qwen2.5-0.5B",    # 한국어 베이스
    "Qwen/Qwen2.5-1.5B",    # 한국어 베이스
    # "mistralai/Mixtral-8x7B-v0.1"  # MoE — 노트북에선 OOM. Colab A100 시도용.
]
for name in models_to_test:
    print(f"\n=== {name} ===")
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16)
    ids = tok(prompt, return_tensors="pt").input_ids
    out = m.generate(ids, max_new_tokens=80, do_sample=False)
    print(tok.decode(out[0], skip_special_tokens=True))
    del m; torch.cuda.empty_cache()

## 실전 — 본 책 10M 모델의 좌표

이 책 본문은 **dense, 10M, decoder-only**만 만든다.

```
파라미터:  ~10M  (모든 dense SLM보다 훨씬 작음)
구조:      decoder-only (BERT 같은 encoder 아님)
형태:      dense (MoE 아님)
도메인:    좁음 (TinyStories 영어 동화)
```

만드는 과정 자체는 dense SLM 1B~7B와 동일 — 같은 nanoGPT 구조, 같은 학습 루프, 같은 평가.

In [ ]:
# 모델 크기 사다리 시각화
landscape = [
    ("본 책 10M",           10,    "Tiny",   "TinyStories 동화"),
    ("SmolLM2-135M",        135,   "SLM",    "Cosmo + FineWeb-Edu"),
    ("SmolLM2-1.7B",        1700,  "SLM",    "11T 토큰 over-train"),
    ("Phi-3-mini",          3800,  "SLM",    "교과서 합성 데이터"),
    ("Qwen2.5-7B",          7000,  "SLM",    "다국어 + 코드"),
    ("Llama 3 8B",          8000,  "SLM",    "15T 토큰"),
    ("Mixtral 8x7B (MoE)",  47000, "Large",  "47B total / 13B active"),
]

print(f"{'모델':22s}  {'파라미터':>10s}  {'등급':>7s}  {'특징'}")
print("-" * 75)
for name, params, tier, note in landscape:
    unit = "M" if params < 1000 else "B"
    val = params if params < 1000 else params / 1000
    marker = " <-- 우리" if "본 책" in name else ""
    print(f"  {name:22s}  {val:7.1f}{unit}  {tier:>7s}  {note}{marker}")

In [ ]:
# 모델 카드 7가지 체크 항목 — SmolLM2-1.7B 기준
# (실제 확인은 HuggingFace 모델 카드에서)

checklist = {
    "전체 파라미터":    "1.7B (dense)",
    "활성 파라미터":    "1.7B (dense이므로 = 전체)",
    "학습 토큰 수":     "11T 토큰 (강한 over-training)",
    "학습 데이터":      "Cosmopedia v2 + FineWeb-Edu + DCLM + The Stack",
    "컨텍스트 길이":    "8192 토큰",
    "라이선스":         "Apache 2.0",
    "토크나이저":       "ByteLevel BPE, vocab 49152",
}

print("SmolLM2-1.7B 모델 카드 요약:")
print("-" * 55)
for k, v in checklist.items():
    print(f"  {k:15s}: {v}")

## 연습문제

1. HuggingFace의 `HuggingFaceTB/SmolLM2-1.7B`와 `Qwen/Qwen2.5-1.5B`의 모델 카드를 읽고 §4의 7가지 항목을 표로 정리하라. 어느 쪽이 한국어에 유리한가?
2. Mixtral 8×7B의 "전체 47B / 활성 13B"는 어떻게 산수가 나오나? 8 expert × 7B면 56B가 나올 텐데. (힌트: shared parameter)
3. 본 책 캡스톤에 쓸 모델로 SmolLM2-1.7B / Qwen 2.5-1.5B / Gemma 2-2B 중 하나를 골라라. **결정 근거 3줄** + **양자화 후 본인 노트북에서 추론 가능한지**.
4. 미래에 (가설) 당신 회사가 자체 SLM을 만든다면 1B / 3B / 7B 중 어느 크기? 디바이스 사다리 + ROI 관점.
5. **(생각해볼 것)** dense와 MoE의 학습 비용 차이는 어디서 오는가? 같은 활성 파라미터 (13B)면 학습 비용도 같을 것 같지만 그렇지 않은 이유.

In [ ]:
# 연습문제 1 — 모델 카드 비교표
items = ["전체 파라미터", "활성 파라미터", "학습 토큰 수", "학습 데이터",
         "컨텍스트 길이", "라이선스", "토크나이저"]

# 직접 모델 카드를 읽고 채워보세요
smollm = {
    "전체 파라미터": "1.7B",
    "활성 파라미터": "1.7B (dense)",
    "학습 토큰 수": "11T",
    "학습 데이터": "Cosmopedia v2 + FineWeb-Edu",
    "컨텍스트 길이": "8192",
    "라이선스": "Apache 2.0",
    "토크나이저": "ByteLevel BPE 49152",
}
qwen = {
    "전체 파라미터": "1.5B",
    "활성 파라미터": "1.5B (dense)",
    "학습 토큰 수": "18T (추정)",
    "학습 데이터": "다국어 + 코드 위주",
    "컨텍스트 길이": "32768",
    "라이선스": "Apache 2.0",
    "토크나이저": "tiktoken-like, 151643",
}

print(f"{'항목':15s}  {'SmolLM2-1.7B':30s}  {'Qwen2.5-1.5B':30s}")
print("-" * 80)
for item in items:
    print(f"  {item:15s}  {smollm[item]:30s}  {qwen[item]:30s}")

print("\n한국어 유리한 쪽: Qwen2.5-1.5B (다국어 학습 비중 높음)")

In [ ]:
# 연습문제 2 — Mixtral 8×7B 파라미터 산수
# 8 expert × 7B = 56B 가 아닌 47B인 이유:
# Transformer 레이어에서 FFN 부분만 expert로 나뉘고,
# Attention 레이어는 shared (모든 토큰이 공유)

# Mistral 7B 기준 구조 (대략)
n_layers = 32
hidden = 4096
ffn_dim = 14336  # Mistral 7B FFN 차원
n_heads = 32

# Attention 파라미터 (shared across all tokens)
attn_params = n_layers * 4 * hidden * hidden  # Q,K,V,O
print(f"Attention params (shared): {attn_params/1e9:.2f}B")

# FFN 파라미터 (per expert)
ffn_per_expert = n_layers * 3 * hidden * ffn_dim  # gate, up, down
print(f"FFN params per expert: {ffn_per_expert/1e9:.2f}B")

# 8 experts
n_experts = 8
total = attn_params + n_experts * ffn_per_expert
print(f"Total (8 experts): ~{total/1e9:.0f}B  (실제 Mixtral은 47B)")
print("\nAttention은 shared → 8×7B=56B 아닌 약 47B")

In [ ]:
# 연습문제 3 — 캡스톤 모델 선택 및 추론 가능성 판단

candidates = {
    "SmolLM2-1.7B":  {"params_b": 1.7, "license": "Apache 2.0", "ko_friendly": False, "int4_gb": 1.0},
    "Qwen2.5-1.5B":  {"params_b": 1.5, "license": "Apache 2.0", "ko_friendly": True,  "int4_gb": 0.9},
    "Gemma2-2B":     {"params_b": 2.0, "license": "Gemma",       "ko_friendly": False, "int4_gb": 1.2},
}

if torch.cuda.is_available():
    my_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    my_vram_gb = 8  # 본인 RAM 또는 VRAM 직접 입력

print(f"본인 메모리: {my_vram_gb:.1f} GB\n")
print(f"{'모델':15s}  {'크기':>6s}  {'라이선스':>12s}  {'한국어':>6s}  {'int4 GB':>8s}  {'추론 가능':>8s}")
print("-" * 70)
for name, info in candidates.items():
    feasible = "YES" if info["int4_gb"] < my_vram_gb else "NO"
    ko = "O" if info["ko_friendly"] else "X"
    print(f"  {name:15s}  {info['params_b']:4.1f}B  {info['license']:>12s}  {ko:>6s}  {info['int4_gb']:>6.1f} GB  {feasible:>8s}")

In [ ]:
# 연습문제 4 & 5 — 자유 메모
